In [1]:
raise RuntimeError(
    "This notebook demonstrates API usage. "
    "Adzuna API credentials are not included for security reasons."
)

RuntimeError: This notebook demonstrates API usage. Adzuna API credentials are not included for security reasons.

In [ ]:
import requests 
import pandas as pd
from datetime import datetime 

In [ ]:
COUNTRY = "in"

In [ ]:
url = f"https://api.adzuna.com/v1/api/jobs/{COUNTRY}/search/1"

In [ ]:
params = {
    "app_id": APP_ID,
    "app_key": APP_KEY,
    "what": "data",
    "results_per_page": 50,
    "content-type": "application/json"
}

In [ ]:
response = requests.get(url, params=params)
print("Status code:", response.status_code)

data = response.json()
print("Total results returned:", len(data.get("results", [])))

In [ ]:
jobs = []

for job in data["results"]:
    jobs.append({
        "job_title": job.get("title"),
        "company": job.get("company", {}).get("display_name"),
        "location": job.get("location", {}).get("display_name"),
        "salary_min": job.get("salary_min"),
        "salary_max": job.get("salary_max"),
        "contract_type": job.get("contract_type"),
        "category": job.get("category", {}).get("label"),
        "job_url": job.get("redirect_url"),
        "created": job.get("created"),
        "scraped_at": datetime.now()
    })

df = pd.DataFrame(jobs)
df.head()


In [ ]:
df.shape
df.columns

In [ ]:
df = df.drop_duplicates(subset=["job_title", "company", "location"])

In [ ]:
df["salary_min"] = df["salary_min"].fillna(0)
df["salary_max"] = df["salary_max"].fillna(0)

In [ ]:
df["company"].value_counts().head(10)

In [ ]:
df["location"].value_counts().head(10)

In [ ]:
df["job_title"].value_counts().head(10)

In [ ]:
df["job_title"].str.lower().value_counts().head(10)

In [ ]:
df.to_csv(
    "adzuna_data_jobs_cleaned_FINAL.csv",
    index=False,
    encoding="utf-8"
)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

df["company"].value_counts().head(10).plot(kind="barh")

plt.title("Top Hiring Companies (Engineering Jobs)")
plt.xlabel("Number of Jobs")
plt.ylabel("Company")

plt.tight_layout()
plt.show()



In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
data_roles = df[
    df["job_title"].str.contains(
        "analyst|analytics|business analyst|data analyst",
        case=False,
        na=False
    )
]

data_roles.shape


In [ ]:
import matplotlib.pyplot as plt

data_roles["location"].value_counts().head(10).plot(kind="barh")
plt.title("Top Locations for Data Analyst Jobs (India)")
plt.xlabel("Number of Jobs")
plt.ylabel("Location")
plt.tight_layout()
plt.show()


In [ ]:
data_roles[["salary_min", "salary_max"]].notnull().sum()

In [ ]:
df.to_csv("adzuna_data_jobs_raw.csv", index=False)
data_roles.to_csv("adzuna_data_analyst_jobs_india.csv", index=False)